<a href="https://colab.research.google.com/github/ust-buan-640/buan-640-notebooks/blob/main/notebooks/hypothesis-testing-inclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hypothesis Testing: In-Class Notebook

Three exercises for tonight, plus one for home. Run the cells in order, from top to bottom, with **Shift + Enter**. Nothing in this notebook is graded, and Python will not be on the tests.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng()

## 1. Back to the Red Bead Factory

A new paddle comes out of the box with **30 red beads out of 50**. The box is still 20% red. Is the process still in control?

### Step 1: estimate p from this paddle

We start with something you already know: a 95% confidence interval for p, the share of red beads the paddle collects.

In [ ]:
n = 50
red = 30
p_hat = red / n

se_ci = np.sqrt(p_hat * (1 - p_hat) / n)
print(f"p-hat = {p_hat:.2f}")
print(f"Standard error for the interval: {se_ci:.4f}")
print(f"95% confidence interval: {p_hat - 1.96 * se_ci:.3f} to {p_hat + 1.96 * se_ci:.3f}")

### Step 2: assume nothing has changed, so p = 0.20

- H₀: p = 0.20 (the process is in control)
- Hₐ: p ≠ 0.20 (the process has changed)

If H₀ is true, how unlikely is a paddle at least as extreme as ours? Notice that the standard error now uses p = 0.20, because we are asking what would happen *if* H₀ were true.

In [ ]:
p0 = 0.20
se_test = np.sqrt(p0 * (1 - p0) / n)
z = (p_hat - p0) / se_test
p_value = 2 * stats.norm.sf(abs(z))

print(f"Standard error if p = 0.20: {se_test:.4f}")
print(f"z = {z:.2f}")
print(f"Two-sided p-value: {p_value:.2e}")

### What if the change is small?

One of Deming's paddles collected red beads at about 18.8%, not 20%. Suppose that is the truth. What is the chance that a two-sided test at α = 0.05, using all the beads from our dips, would reject H₀? That chance is the **power** of the test: P(reject H₀ | H₀ is false).

In [ ]:
def power_two_sided(dips, p_true=0.188, p0=0.20, alpha=0.05):
    n_beads = dips * 50
    se0 = np.sqrt(p0 * (1 - p0) / n_beads)
    se1 = np.sqrt(p_true * (1 - p_true) / n_beads)
    zc = stats.norm.ppf(1 - alpha / 2)
    lower, upper = p0 - zc * se0, p0 + zc * se0          # reject H0 if p-hat falls outside these
    return stats.norm.cdf((lower - p_true) / se1) + stats.norm.sf((upper - p_true) / se1)

dips = 24   # the number of dips in our game
print(f"Power with {dips} dips: {power_two_sided(dips):.1%}")

d = np.arange(5, 301)
plt.figure(figsize=(8, 4))
plt.plot(d, [power_two_sided(k) for k in d])
plt.axhline(0.8, color="gray", linestyle="--")
plt.axvline(dips, color="red", linestyle=":")
plt.xlabel("Number of paddle dips"); plt.ylabel("Power")
plt.title("Chance of detecting a true red share of 18.8% (two-sided, α = 0.05)")
plt.show()

needed = next(k for k in d if power_two_sided(k) >= 0.8)
print(f"Dips needed for 80% power: about {needed}")

**Check it by simulation.** Run our experiment 10,000 times with paddles that really do collect red beads at 18.8%, and count how often the test rejects H₀.

In [ ]:
sims = 10_000
n_beads = dips * 50
reds = rng.binomial(n_beads, 0.188, size=sims)
z_sim = (reds / n_beads - 0.20) / np.sqrt(0.20 * 0.80 / n_beads)
print(f"Share of simulated experiments that reject H0: {np.mean(np.abs(z_sim) > stats.norm.ppf(0.975)):.1%}")

## 2. Data of Homes: is the average list price $380,000?

A realtor says that the average list price in our zip codes is $380,000. Test the claim at α = 0.05, treating our 93 homes as an independent random sample. Work in pairs, and write each step on the handout.

You may use Colab's built-in AI to help with the code, for example to compute the standard deviation or to find the critical t value. Write the steps and the conclusion yourself.

First, load the data.

In [ ]:
url = "https://raw.githubusercontent.com/ust-buan-640/buan-640-notebooks/main/data/data_of_homes_f26.csv"
df = pd.read_csv(url)
price = df["list_price"]
print("Number of homes:", len(price))

**Step 0.** Which test should we use, and why? Are its conditions met?

**Step 1.** State H₀ and Hₐ.

**Step 2.** Compute the sample mean x̄ and the sample standard deviation s.

In [ ]:
# Step 2: compute the sample mean and the sample standard deviation of price

**Step 3.** Compute the standard error s/√n and the test statistic t = (x̄ − 380,000) / (s/√n).

In [ ]:
# Step 3: compute the standard error and the test statistic

**Step 4.** Find the critical t value for a two-sided test at α = 0.05 (how many degrees of freedom?), and the p-value.

In [ ]:
# Step 4: find the critical value and the p-value

**Step 5.** Decide, and write the conclusion in one sentence a realtor would understand.

### Check your work

The function `ttest_1samp` does Steps 2–4 at once. It uses a two-sided alternative by default.

In [ ]:
check = stats.ttest_1samp(price, popmean=380_000)
print(f"t = {check.statistic:.3f}, two-sided p-value = {check.pvalue:.4f}")
print(f"Critical value with {len(price) - 1} degrees of freedom: {stats.t.ppf(0.975, df=len(price) - 1):.3f}")

### But were the 93 homes independent?

The sample was collected in two stages: seven zip codes were drawn at random, and then about 13 homes were sampled in each. Homes in the same zip code resemble each other, so the 93 homes are not 93 independent observations.

A simple correction is to treat each zip code as one observation, and run the same test on the seven zip code means.

In [ ]:
zip_means = df.groupby("zip")["list_price"].mean()
k = len(zip_means)

x_bar_z = zip_means.mean()
s_z = zip_means.std()
se_z = s_z / np.sqrt(k)
t_z = (x_bar_z - 380_000) / se_z
t_crit_z = stats.t.ppf(0.975, df=k - 1)
p_z = 2 * stats.t.sf(abs(t_z), df=k - 1)

print(f"Step 2: mean of the zip means = {x_bar_z:,.0f}, s = {s_z:,.0f}, n = {k}")
print(f"Step 3: standard error = {se_z:,.0f}, t = {t_z:.3f}")
print(f"Step 4: critical value with {k - 1} degrees of freedom = {t_crit_z:.3f}, p-value = {p_z:.3f}")

### Same data, opposite conclusions

Compare the two analyses side by side, together with their 95% confidence intervals.

In [ ]:
ci_naive = stats.t.interval(0.95, df=len(price) - 1, loc=price.mean(), scale=stats.sem(price))
ci_zip = stats.t.interval(0.95, df=k - 1, loc=x_bar_z, scale=se_z)

summary = pd.DataFrame({
    "t": [check.statistic, t_z],
    "p-value": [check.pvalue, p_z],
    "95% interval": [f"{ci_naive[0]:,.0f} to {ci_naive[1]:,.0f}", f"{ci_zip[0]:,.0f} to {ci_zip[1]:,.0f}"],
}, index=["93 homes treated as independent", "Seven zip code means"])
summary.round(3)

**With your partner:** each test rejects H₀ exactly when $380,000 falls outside its interval. Check this in the table. Then rewrite your sentence for the realtor using the corrected result.

*Your sentence:*

## 3. What is a significant result worth?

### 3a. The mammogram

The numbers below are rounded from US screening data for women in their 40s: about 3 in 1,000 have breast cancer, the mammogram catches about 80% of cancers, and about 12% of women without cancer are called back anyway. Change the numbers and see how the answer moves. Which of the three inputs matters most?

In [ ]:
def screening(prevalence, sensitivity, false_positive_rate, people=1000):
    sick = people * prevalence
    healthy = people - sick
    true_pos = sick * sensitivity
    false_pos = healthy * false_positive_rate
    print(f"Out of {people:,} people: {sick:,.0f} have the condition, {healthy:,.0f} do not.")
    print(f"Positive tests: {true_pos:,.1f} true positives and {false_pos:,.1f} false positives.")
    print(f"P(condition | positive) = {true_pos / (true_pos + false_pos):.1%}")

screening(prevalence=0.003, sensitivity=0.80, false_positive_rate=0.12, people=10_000)

### 3b. The same tree for research

Out of all the hypotheses researchers test, some share is actually true. What fraction of significant results are real?

In [ ]:
def share_real(prior_true, power, alpha=0.05, hypotheses=1000):
    true_h = hypotheses * prior_true
    false_h = hypotheses - true_h
    sig_true = true_h * power
    sig_false = false_h * alpha
    return sig_true / (sig_true + sig_false)

print(f"Prior 10%, power 0.5: {share_real(0.10, 0.5):.0%} of significant results are real")

table = pd.DataFrame(
    {f"power {pw}": [share_real(pr, pw) for pr in (0.01, 0.05, 0.10, 0.25, 0.50)] for pw in (0.2, 0.5, 0.8)},
    index=["prior 1%", "prior 5%", "prior 10%", "prior 25%", "prior 50%"],
)
table.style.format("{:.0%}")

### 3c. Did the sun just explode?

In the xkcd comic, the detector lies only when two dice both come up six, so P(YES | sun has not exploded) = 1/36 and P(YES | sun has exploded) = 35/36. Pick a prior probability that the sun has just exploded, and compute P(sun exploded | YES).

In [ ]:
prior = 1e-9   # your guess for the probability that the sun has just exploded

p_yes = (35/36) * prior + (1/36) * (1 - prior)
print(f"P(sun exploded | YES) = {(35/36) * prior / p_yes:.2e}")

## 4. For home: jelly beans cause acne?

A research team suspects that jelly beans cause acne. In the general population, 20% of people have acne. The team recruits 50 people for each of 20 colors of jelly beans and tests, color by color, whether the share with acne is above 20%.

In this simulation, jelly beans have **no effect at all**: every group has a 20% acne rate. Run the cell several times.

In [ ]:
colors = ["purple", "brown", "pink", "blue", "teal", "salmon", "red", "turquoise", "magenta", "yellow",
          "grey", "tan", "cyan", "green", "mauve", "beige", "lilac", "black", "peach", "orange"]
n_per = 50
acne = rng.binomial(n_per, 0.20, size=len(colors))
z_col = (acne / n_per - 0.20) / np.sqrt(0.20 * 0.80 / n_per)
p_col = 1 - stats.norm.cdf(z_col)

for c, k, p in zip(colors, acne, p_col):
    flag = "  <-- significant!" if p < 0.05 else ""
    print(f"{c:10s} {k:2d} of 50 with acne, p-value = {p:.3f}{flag}")

How often does at least one color come out significant, when there is no effect at all? Simulate the whole study 10,000 times.

In [ ]:
studies = 10_000
acne = rng.binomial(n_per, 0.20, size=(studies, len(colors)))
p_all = 1 - stats.norm.cdf((acne / n_per - 0.20) / np.sqrt(0.20 * 0.80 / n_per))
print(f"Studies with at least one 'significant' color: {np.mean((p_all < 0.05).any(axis=1)):.0%}")

**Questions.** If only the significant color gets reported in the news, what should you conclude from the headline? How does this relate to the research tree in Section 3b? (The simulated share comes out around 70%, a bit above the 64% you might expect from 1 − 0.95²⁰. With only 50 people per group the normal approximation is slightly off, and this test actually rejects a true H₀ about 6% of the time rather than 5%.)